# COMP4328 Assignment 2 — experiments and analysis

This notebook uses the Python modules in `algorithm/`. It does not mount or modify Drive. Read `README.md` and `METHODS.md` for equations, assumptions and the experiment protocol. Smoke results cannot be used as final results.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

candidates = [Path.cwd(), Path.cwd() / "Assignment" / "solution", Path.cwd().parent]
ROOT = next((p.resolve() for p in candidates if (p / "algorithm" / "main.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the extracted solution folder")
DATA_DIR = ROOT / "data"
if not any(DATA_DIR.glob("*.npz")) and (ROOT.parent / "review" / "datasets").is_dir():
    DATA_DIR = ROOT.parent / "review" / "datasets"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Source:", ROOT)
print("Data:", DATA_DIR)


## Configuration

The predefined extended suite uses ten random splits, the original base epoch budgets, matched refinement controls and both estimator strategies. Keep training flags false when inspecting saved results. Existing outputs require an explicit resume with identical code/configuration/environment. In Colab, first upload/extract this package and place the datasets in local `data/`; save a download of your outputs before the runtime expires.

In [ ]:
RUN_SMOKE = False
RUN_FINAL = False
RESUME = False
DEVICE = "auto"
FINAL_ROOT = ROOT / "experiments" / "final_v1"
SMOKE_ROOT = ROOT / "verification" / "notebook_smoke_v1"
ANALYSE_ROOT = FINAL_ROOT


## Separate smoke check

Two small runs exercise all datasets, estimator modes and methods. They are explicitly non-reportable and cannot overwrite the final directory.

In [ ]:
if RUN_SMOKE:
    command = [sys.executable, str(ROOT / "algorithm" / "suite.py"), "--data-dir", str(DATA_DIR), "--output", str(SMOKE_ROOT), "--device", DEVICE, "--smoke", "--runs", "2", "--epochs", "2", "--sample-limit", "600", "--test-limit", "300"]
    if RESUME:
        command.append("--resume")
    subprocess.run(command, check=True)
else:
    print("Smoke training disabled")


## Final experiment suite

This can take hours on CPU. No hyperparameter or checkpoint selection uses clean test accuracy.

In [ ]:
if RUN_FINAL:
    command = [sys.executable, str(ROOT / "algorithm" / "suite.py"), "--data-dir", str(DATA_DIR), "--output", str(FINAL_ROOT), "--device", DEVICE]
    if RESUME:
        command.append("--resume")
    subprocess.run(command, check=True)
else:
    print("Final training disabled")


## Verified tables and matrices

Only completed run records enter a summary. Accuracy uncertainty is sample SD across splits; a single-run SD is undefined. The export labels smoke/partial output. For the final report, include the actual CIFAR estimate and its uncertainty and explain known-T versus estimated-T revision separately.

In [ ]:
from algorithm.analyse import experiment_files, export
files = experiment_files(ANALYSE_ROOT)
if files:
    analysis_dir = ROOT / "analysis" / ANALYSE_ROOT.name
    rows = export(ANALYSE_ROOT, analysis_dir)
    text = (analysis_dir / "tables.md").read_text(encoding="utf-8")
    try:
        from IPython.display import Markdown, display
        display(Markdown(text))
    except ImportError:
        print(text)
    for path in files:
        result = json.loads(path.read_text(encoding="utf-8"))
        print(path.parent.name, "estimated T:", result["summary"]["T_hat"], "revised T:", result["summary"]["T_revised"])
else:
    print("No results yet at", ANALYSE_ROOT)


## Optional figures and checkpoint replay

Install `requirements-analysis.txt` for figures. Checkpoint replay verifies every saved accuracy against predictions on the same test examples; it does not retrain or select models.

In [ ]:
MAKE_PLOTS = False
REPLAY_CHECKPOINTS = False
if files and MAKE_PLOTS:
    export(ANALYSE_ROOT, ROOT / "analysis" / ANALYSE_ROOT.name, plots=True)
if files and REPLAY_CHECKPOINTS:
    subprocess.run([sys.executable, str(ROOT / "algorithm" / "verify.py"), "--root", str(ANALYSE_ROOT), "--data-dir", str(DATA_DIR), "--replay", "--output", str(ROOT / "analysis" / (ANALYSE_ROOT.name + "_audit.json"))], check=True)
